# 04. XAI(SHAP) 분석 - LEED 등급 결정 영향 요인 도출

## 핵심 연구 결과물

> **본 노트북이 본 연구의 핵심 기여입니다.**
> 
> 기존 연구: "Gold 등급 건물들은 평균적으로 에너지 점수가 높더라" (단순 통계)
> 
> **본 연구 (SHAP)**:
> - "연면적 3만㎡ 이상 오피스 건물이 EA 점수 18점을 받았을 때 Gold 등급에 미치는 영향" = +0.43
> - "WE 점수가 8점 이상일 때 Platinum 등급에 기여하는 정도" = +0.31
> - 등급 결정 시 가장 중요한 카테고리 TOP 3 순위

## SHAP 시각화 종류
1. **Feature Importance Bar** - 전체 영향력 순위
2. **Summary Plot (Beeswarm)** - 점수 수준별 영향 방향
3. **Dependence Plot** - 특정 카테고리 점수 ↔ SHAP 관계
4. **Grade Comparison Boxplot** - 등급별 영향력 비교
5. **Force Plot** - 개별 건물 설명

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import shap
import joblib
import os
from pathlib import Path

matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

from src.data.loader import LEEDDataLoader
from src.data.preprocessor import LEEDPreprocessor, GRADE_ENCODING
from src.analysis.ml_models import LEEDMLTrainer
from src.analysis.xai_shap import LEEDSHAPAnalyzer

GRADE_LABELS = {v: k for k, v in GRADE_ENCODING.items()}
print('라이브러리 로딩 완료')

## 1. 데이터 및 모델 로딩

In [ ]:
preprocessor = LEEDPreprocessor()
trainer = LEEDMLTrainer(output_dir='../outputs')

# ─────────────────────────────────────────────────────────────────────
# 03번 노트북에서 저장한 표준화 데이터 & 모델 로딩
# ─────────────────────────────────────────────────────────────────────
standardized_path = '../data/processed/korea_leed_standardized_v5.csv'
model_path = '../outputs/leed_model.pkl'

if os.path.exists(standardized_path):
    df = pd.read_csv(standardized_path)
    print(f'데이터 로딩: {len(df)}개 프로젝트')
else:
    print('[주의] 표준화 데이터 없음. 새로 생성...')
    loader = LEEDDataLoader()
    raw_df = loader.create_sample_data()
    df = preprocessor.run_pipeline(raw_df)

# grade_encoded, log_area, type_ 컬럼 보정
if 'grade_encoded' not in df.columns:
    df = preprocessor.encode_grade(df)
if 'log_area' not in df.columns:
    df = preprocessor.log_transform_area(df)
if not any(c.startswith('type_') for c in df.columns):
    df = preprocessor.encode_building_type(df)

X, y = preprocessor.split_features_target(df)
feature_names = list(X.columns)

In [ ]:
# 모델 로딩 or 재학습
if os.path.exists(model_path):
    trainer.load_model('leed_model.pkl')
    model = trainer.best_model
    print(f'저장된 모델 로딩: {trainer.best_model_name}')
else:
    print('[주의] 모델 없음. 새로 학습...')
    trainer.evaluate_models(X, y, cv_folds=5)
    model = trainer.train(X, y)
    trainer.save_model('leed_model.pkl')

## 2. SHAP 값 계산

In [ ]:
analyzer = LEEDSHAPAnalyzer(
    model=model,
    feature_names=feature_names,
    output_dir='../outputs/figures'
)

shap_values = analyzer.compute_shap_values(X)

## 3. 전체 특성 중요도 (핵심 결과)

어떤 카테고리가 LEED 등급 결정에 가장 큰 영향을 미치는가?

In [ ]:
importance_df = analyzer.plot_feature_importance(save=True)
print('\nLEED 등급 결정 영향력 순위 (전체 등급 통합):')
print(importance_df.to_string(index=False))

## 4. 등급별 Summary Plot

각 카테고리 점수 수준(높음/낮음)이 해당 등급에 어떻게 영향을 주는가?

In [ ]:
# Gold 등급 분석 (가장 많은 한국 건물이 목표로 하는 등급)
print('=== Gold 등급 영향 요인 ===')
analyzer.plot_summary(class_idx=2, save=True)  # 2 = Gold

In [ ]:
# Platinum 등급 분석
print('=== Platinum 등급 영향 요인 ===')
analyzer.plot_summary(class_idx=3, save=True)  # 3 = Platinum

## 5. 에너지(EA) 의존성 분석

EA 점수가 높아질수록 Gold 등급 획득 가능성이 어떻게 변하는가?
건물 유형(오피스/주거)에 따라 차이가 있는가?

In [ ]:
if 'score_v5_EA' in feature_names:
    analyzer.plot_dependence(
        feature='score_v5_EA',
        class_idx=2,  # Gold
        interaction_feature='auto',
        save=True
    )
    
    # WE(물 효율) 의존성도 분석
if 'score_v5_WE' in feature_names:
    analyzer.plot_dependence(
        feature='score_v5_WE',
        class_idx=2,  # Gold
        save=True
    )

## 6. 등급별 영향력 비교

같은 EA 점수여도 Certified vs Platinum 등급에서 영향력이 다른가?

In [ ]:
if 'score_v5_EA' in feature_names:
    analyzer.plot_grade_comparison(
        feature='score_v5_EA',
        y=y,
        save=True
    )

if 'score_v5_SS' in feature_names:
    analyzer.plot_grade_comparison(
        feature='score_v5_SS',
        y=y,
        save=True
    )

## 7. 개별 건물 Force Plot

특정 건물이 Gold 등급을 받은 이유를 각 카테고리별로 분해

In [ ]:
# Gold 등급 건물 1개 선택
gold_mask = y == GRADE_ENCODING.get('Gold', 2)
if gold_mask.sum() > 0:
    gold_idx = gold_mask[gold_mask].index[0]
    sample_pos = list(X.index).index(gold_idx)
    
    print(f'분석 대상 건물 (index: {gold_idx}):')
    print(X.iloc[sample_pos])
    
    analyzer.plot_force(sample_idx=sample_pos, class_idx=2, save=True)

## 8. 종합 SHAP 분석 리포트 생성

논문의 핵심 결과표: 카테고리별 등급별 평균 SHAP 값

In [ ]:
report = analyzer.generate_report(y)
print('\nSHAP 종합 분석 리포트:')
display(report)

## 9. 연구 결론 요약

SHAP 분석 결과를 바탕으로 한국 LEED 인증 건물의 등급 결정 주요 요인 정리

In [ ]:
# 상위 3개 중요 카테고리 추출
top3 = importance_df.head(3)

print('=' * 60)
print('  연구 결과 요약: 한국 LEED 등급 결정 핵심 요인')
print('=' * 60)
print('\nXAI(SHAP) 분석 결과 TOP 3 영향 카테고리:')
for i, (_, row) in enumerate(top3.iterrows(), 1):
    print(f'  {i}위: {row["Feature"]} (평균 |SHAP|: {row["Mean_SHAP"]:.4f})')

print('\n연구 차별성:')
print('  - 기존: 단순 평균 비교 → 등급 예측')
print('  - 본 연구: SHAP 값으로 개별 카테고리의 실제 영향력 정량화')
print('  - 한국 맞춤형 가성비 높은 LEED 설계 전략 도출 가능')

print('\n모든 분석 완료. outputs/ 폴더에서 결과를 확인하세요.')